<a id='setup'></a>
## 1. Setup & Imports

In [ ]:
# Install required packages (if needed)
# !pip install datasets transformers torch mlflow scikit-learn matplotlib seaborn streamlit

In [ ]:
# Standard imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

<a id='data'></a>
## 2. Data Loading & Exploratory Data Analysis

In [ ]:
# Add src to path
sys.path.append('../src')

from data_prep import AGNewsDataPrep

# Initialize data preparation
data_prep = AGNewsDataPrep(cache_dir="../data")

# Load data
data_prep.load_data()

In [ ]:
# Get basic statistics
stats = data_prep.get_basic_stats()
print("Dataset Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Analyze class distribution
data_prep.analyze_class_distribution(save_fig=True)

In [ ]:
# Analyze text lengths
data_prep.analyze_text_lengths(save_fig=True)

In [ ]:
# Show sample articles
data_prep.show_sample_articles(n_samples=2)

In [ ]:
# Vocabulary statistics
vocab_stats = data_prep.get_vocabulary_stats()

In [ ]:
# Word frequency analysis
data_prep.plot_word_frequency(top_n=20, save_fig=True)

<a id='baseline'></a>
## 3. Baseline Model Training (TF-IDF + Logistic Regression)

In [ ]:
from train_baseline import train_with_mlflow as train_baseline

# Train baseline model with MLflow tracking
baseline_model, baseline_metrics = train_baseline()

In [ ]:
# Display baseline results
print("\nBaseline Model Results:")
print("="*50)
for metric, value in baseline_metrics.items():
    if 'time' in metric:
        print(f"{metric}: {value:.2f}ms" if value < 1000 else f"{metric}: {value/1000:.2f}s")
    else:
        print(f"{metric}: {value:.4f}")

<a id='transformer'></a>
## 4. Transformer Model Training (DistilBERT)

⚠️ **Note:** This cell may take 20-30 minutes to run depending on your hardware.

In [ ]:
from train_transformer import train_with_mlflow as train_transformer

# Train transformer model with MLflow tracking
transformer_model, transformer_metrics = train_transformer()

In [ ]:
# Display transformer results
print("\nTransformer Model Results:")
print("="*50)
for metric, value in transformer_metrics.items():
    if 'time' in metric:
        print(f"{metric}: {value:.2f}ms" if value < 1000 else f"{metric}: {value/1000:.2f}s")
    else:
        print(f"{metric}: {value:.4f}")

<a id='evaluation'></a>
## 5. Model Evaluation & Comparison

In [ ]:
from evaluate import ModelEvaluator

# Initialize evaluator
evaluator = ModelEvaluator()

# Compare models
comparison_df = evaluator.compare_models(baseline_metrics, transformer_metrics)
print("\n", comparison_df)

In [ ]:
# Plot model comparison
evaluator.plot_model_comparison(baseline_metrics, transformer_metrics)

### View MLflow Experiments

To view detailed experiment tracking:

```bash
mlflow ui
```

Then open: http://localhost:5000

<a id='inference'></a>
## 6. Inference Examples

In [ ]:
from inference import BaselinePredictor, TransformerPredictor

# Load predictors
baseline_predictor = BaselinePredictor()
baseline_predictor.load_model()

transformer_predictor = TransformerPredictor()
transformer_predictor.load_model()

print("✓ Models loaded for inference")

In [ ]:
# Test examples
test_examples = [
    "Lakers defeat Celtics in overtime thriller to win NBA championship",
    "Federal Reserve announces interest rate hike to combat inflation",
    "New quantum computer breakthrough could revolutionize cryptography",
    "UN Security Council holds emergency meeting on global crisis"
]

print("INFERENCE EXAMPLES")
print("="*70)

for i, text in enumerate(test_examples, 1):
    print(f"\nExample {i}: {text}")
    print("-"*70)
    
    # Baseline prediction
    baseline_cat, baseline_conf = baseline_predictor.predict(text)
    print(f"Baseline: {baseline_cat} (confidence: {baseline_conf[baseline_cat]:.4f})")
    
    # Transformer prediction
    transformer_cat, transformer_conf = transformer_predictor.predict(text)
    print(f"Transformer: {transformer_cat} (confidence: {transformer_conf[transformer_cat]:.4f})")

In [ ]:
# Interactive prediction
def predict_article(text):
    """Predict category for custom text"""
    print(f"\nInput: {text}")
    print("="*70)
    
    # Baseline
    baseline_cat, baseline_conf = baseline_predictor.predict(text)
    print(f"\nBaseline Model: {baseline_cat}")
    print("Confidence scores:")
    for cat, score in sorted(baseline_conf.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {score:.4f}")
    
    # Transformer
    transformer_cat, transformer_conf = transformer_predictor.predict(text)
    print(f"\nTransformer Model: {transformer_cat}")
    print("Confidence scores:")
    for cat, score in sorted(transformer_conf.items(), key=lambda x: x[1], reverse=True):
        print(f"  {cat}: {score:.4f}")

# Try your own text
custom_text = "Apple reports record quarterly earnings driven by strong iPhone sales"
predict_article(custom_text)

## 🎉 Complete!

### Next Steps

1. **View MLflow UI**: `mlflow ui` to see all experiments
2. **Launch Streamlit App**: `streamlit run ../app/streamlit_app.py`
3. **Explore Results**: Check the `figures/` directory for visualizations
4. **Try Custom Text**: Use the interactive prediction function above

### Project Structure

```
✓ Data loaded and analyzed
✓ Baseline model trained
✓ Transformer model trained
✓ Models evaluated and compared
✓ Ready for deployment
```

### Documentation

- `README.md` - Project overview
- `crisp_dm.md` - Complete CRISP-DM methodology
- `slides_outline.md` - Presentation structure
- `demo_script.md` - Demo video guide